# 05 Sequence Dataset

**Phase 4 — Sequential Learning Analytics**  
Research contract: `PHASE4_RESEARCH_CONTRACT_v1.md`  
Schema version: `seq_v1`

This notebook converts flat snapshot CSVs into ordered learning sequences
suitable for LSTM, GRU, and TAG construction.  
It does **not** connect to Supabase. All inputs are offline CSV snapshots.

## Pipeline

```
Raw snapshot CSVs
  sequence_<date>_<batch>.csv   ← vw_dataset_sequence_level
  attempt_<date>_<batch>.csv    ← vw_dataset_attempt_level
  outcome_<date>_<batch>.csv    ← computed 2C3L labels
        │
        ▼
  [1] Load + validate snapshots
  [2] Canonicalize events (deduplicate client/server pairs)
  [3] Compute cutoff_timestamp per learner
  [4] Anti-leakage check — no post-cutoff features
  [5] Build per-step feature vectors
  [6] Validate labels (label_source / label_validity)
       └─ fallback: behavioral proxy when no rubric labels exist
  [7] Student-level split (GroupShuffleSplit, frozen ledger)
  [8] Build sequence index and canonical events artifact
  [9] Pad + mask sequences → tensors
 [10] Write artifacts + sequence manifest
 [11] Validation summary
```

> ⚠️ **TECHNICAL VALIDATION ONLY** — generated from a mock dataset.  
> Not suitable for research conclusions. Final thesis requires ≥60 participants.

## 0. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)

RAW_DIR  = Path('data/raw')
SEQ_DIR  = Path('data/sequences')
SEQ_DIR.mkdir(parents=True, exist_ok=True)

# Pin specific files to reproduce a run; None = auto-select newest
SEQUENCE_CSV : str | None = None
ATTEMPT_CSV  : str | None = None
OUTCOME_CSV  : str | None = None

# ── Research contract parameters (PHASE4_RESEARCH_CONTRACT_v1.md) ─────────────
SCHEMA_VERSION          = 'seq_v1'
PHASE3_SOURCE_SHA       = '193b18949e40e6bd3bbfb70034a5772ce51d1b7e'
RANDOM_STATE            = 42
TEST_SIZE               = 0.20
AT_RISK_THRESHOLD       = 65.0
DEDUP_WINDOW_SEC        = 5
DUPLICATE_EVENT_TYPES   = {'sql_run', 'submit_answer'}
MAX_SEQ_LEN_PERCENTILE  = 95
VALID_LABEL_SOURCES     = {'teacher_reviewed', 'expert_validated', 'auto_scored_validated'}
# pilot_only: valid for pipeline validation; NOT valid for thesis conclusions
PILOT_LABEL_SOURCES     = {'auto_generated', 'proxy_behavioral'}

CANONICAL_CRITERIA = [
    'c1_correctness_result', 'c2_semantic_consistency',
    'l1_logical_reasoning',  'l2_learning_process', 'l3_difficulty_complexity',
]
LEAKAGE_COLS = (
    set(CANONICAL_CRITERIA)
    | {f'{k}_score' for k in CANONICAL_CRITERIA}
    | {f'{k}_max'   for k in CANONICAL_CRITERIA}
    | {'total_2c3l_score', 'at_risk', 'is_correct_final', 'score_final',
       'rubric_applied_version', 'grade_letter'}
)

# Phase 5 M5.2: block event type sets
# ACTIVE (Phase 5): emitted by BlockSqlBuilder via POST /api/student/block-event
# RESERVED: block_submit is not activated (PHASE5_BLOCK_EVENT_CONTRACT_v1.md §3)
BLOCK_EVENT_ACTIVE_TYPES   = {'block_add', 'block_move', 'block_delete'}
BLOCK_EVENT_RESERVED_TYPES = {'block_submit'}
# Legacy alias kept for backward-compatibility with any downstream reference
BLOCK_EVENT_TYPES = BLOCK_EVENT_ACTIVE_TYPES | BLOCK_EVENT_RESERVED_TYPES

print(f'Schema version    : {SCHEMA_VERSION}')
print(f'Phase 3 source    : {PHASE3_SOURCE_SHA}')
print(f'AT_RISK_THRESHOLD : {AT_RISK_THRESHOLD} (canonical 2C3L threshold)')
print(f'Random state      : {RANDOM_STATE}')
print(f'Block events      : active={sorted(BLOCK_EVENT_ACTIVE_TYPES)}  reserved={sorted(BLOCK_EVENT_RESERVED_TYPES)}')

## 1. Load and validate snapshots

In [2]:
def newest_matching(pattern: str) -> Path | None:
    files = sorted(RAW_DIR.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    return files[0] if files else None

seq_path     = Path(SEQUENCE_CSV) if SEQUENCE_CSV else newest_matching('sequence_*.csv')
attempt_path = Path(ATTEMPT_CSV)  if ATTEMPT_CSV  else newest_matching('attempt_*.csv')
outcome_path = Path(OUTCOME_CSV)  if OUTCOME_CSV  else newest_matching('outcome_*.csv')

for label, p in [('sequence', seq_path), ('attempt', attempt_path), ('outcome', outcome_path)]:
    if p is None:
        raise FileNotFoundError(
            f'No {label} CSV found in {RAW_DIR}. '
            f'Run: node scripts/export-research-snapshots.mjs --batch <BATCH_CODE>'
        )
    print(f'{label:10s}: {p}')

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    h.update(p.read_bytes())
    return h.hexdigest()[:16]

seq_df     = pd.read_csv(seq_path,     encoding='utf-8-sig')
attempt_df = pd.read_csv(attempt_path, encoding='utf-8-sig')
outcome_df = pd.read_csv(outcome_path, encoding='utf-8-sig')

print(f'\nsequence rows : {len(seq_df):,}  |  columns : {seq_df.shape[1]}')
print(f'attempt rows  : {len(attempt_df):,}  |  columns : {attempt_df.shape[1]}')
print(f'outcome rows  : {len(outcome_df):,}  |  columns : {outcome_df.shape[1]}')

sequence  : data\raw\sequence_20260715_MOCK-VALID3-20260715.csv
attempt   : data\raw\attempt_20260715_MOCK-VALID3-20260715.csv
outcome   : data\raw\outcome_20260715_MOCK-VALID3-20260715.csv

sequence rows : 702  |  columns : 13
attempt rows  : 351  |  columns : 9
outcome rows  : 81  |  columns : 24


In [3]:
REQ_SEQ     = {'academy_member_id','batch_code','task_code','session_id','event_order','event_type','event_time'}
REQ_ATTEMPT = {'academy_member_id','batch_code','task_code','attempt_no','is_correct','created_at'}
REQ_OUTCOME = {'participant_code','batch_code','task_code','submission_id','submitted_at',
               'total_2c3l_score','at_risk','label_source','label_validity'}

def check_required(df: pd.DataFrame, required: set, name: str) -> None:
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f'{name} missing required columns: {missing}')
    print(f'  {name:12s} required columns: OK')

check_required(seq_df,     REQ_SEQ,     'sequence')
check_required(attempt_df, REQ_ATTEMPT, 'attempt')
check_required(outcome_df, REQ_OUTCOME, 'outcome')

if 'participant_code' in outcome_df.columns and 'academy_member_id' not in outcome_df.columns:
    outcome_df = outcome_df.rename(columns={'participant_code': 'academy_member_id'})

seq_df['event_time']    = pd.to_datetime(seq_df['event_time'],    utc=True, errors='coerce')
attempt_df['created_at']= pd.to_datetime(attempt_df['created_at'],utc=True, errors='coerce')
if 'submitted_at' in outcome_df.columns:
    outcome_df['submitted_at'] = pd.to_datetime(outcome_df['submitted_at'], utc=True, errors='coerce')

print('\nTimestamp parsing: OK')

  sequence     required columns: OK
  attempt      required columns: OK
  outcome      required columns: OK

Timestamp parsing: OK


## 2. Canonicalize events — deduplicate client/server pairs

Both `sql_run` and `submit_answer` are fired by both the client page (optimistic)
and the server route handler.

**Rule (research contract §8):** Retain the event with the higher `event_order`
(server-side); drop the lower (client-side).

In [4]:
seq_df = seq_df.sort_values(['session_id','event_order']).reset_index(drop=True)

def mark_client_duplicates(df: pd.DataFrame, dup_types: set, window_sec: int) -> pd.Series:
    is_dup = pd.Series(False, index=df.index)
    cands  = df[df['event_type'].isin(dup_types)].copy()
    if cands.empty:
        return is_dup
    cands['prev_session']    = cands.groupby('session_id')['session_id'].shift(1)
    cands['prev_event_type'] = cands.groupby('session_id')['event_type'].shift(1)
    cands['prev_order']      = cands.groupby('session_id')['event_order'].shift(1)
    cands['prev_time']       = cands.groupby('session_id')['event_time'].shift(1)
    cands['delta_order']     = cands['event_order'] - cands['prev_order']
    cands['delta_sec']       = (cands['event_time'] - cands['prev_time']).dt.total_seconds().abs()
    pair_mask = (
        (cands['prev_session']    == cands['session_id']) &
        (cands['prev_event_type'] == cands['event_type']) &
        (cands['delta_order']     == 1) &
        (cands['delta_sec']       <= window_sec)
    )
    prev_indices = cands.index[pair_mask] - 1
    is_dup.iloc[prev_indices[prev_indices >= 0]] = True
    return is_dup

dup_mask = mark_client_duplicates(seq_df, DUPLICATE_EVENT_TYPES, DEDUP_WINDOW_SEC)
seq_df['dropped_as_duplicate'] = dup_mask
canonical_df = seq_df[~dup_mask].copy()
n_dropped    = int(dup_mask.sum())
block_found  = int(canonical_df['event_type'].isin(BLOCK_EVENT_TYPES).sum())

print(f'Raw events     : {len(seq_df):,}')
print(f'Dropped (dups) : {n_dropped:,}  (window={DEDUP_WINDOW_SEC}s)')
print(f'Canonical      : {len(canonical_df):,}')
print(f'Block events   : {block_found}  (expected 0 — BlockSqlBuilder not connected)')

Raw events     : 702
Dropped (dups) : 0  (window=5s)
Canonical      : 702
Block events   : 0  (expected 0 — BlockSqlBuilder not connected)


## 3. Compute cutoff_timestamp per learner × task

In [5]:
submit_events = canonical_df[canonical_df['event_type'] == 'submit_answer'].copy()
cutoff_df = (
    submit_events.sort_values('event_time')
    .groupby(['academy_member_id','task_code'], as_index=False).first()
    [['academy_member_id','task_code','event_time']]
    .rename(columns={'event_time': 'cutoff_timestamp'})
)
print(f'Learner × task cutoffs computed: {len(cutoff_df)}')
print(cutoff_df.head())

canonical_df = canonical_df.merge(cutoff_df, on=['academy_member_id','task_code'], how='left')
canonical_df['is_post_cutoff'] = (
    canonical_df['event_time'] >= canonical_df['cutoff_timestamp']
).where(canonical_df['cutoff_timestamp'].notna(), other=False)

pre_cutoff_df = canonical_df[~canonical_df['is_post_cutoff']].copy()
print(f'\nPre-cutoff events  : {len(pre_cutoff_df):,}')
print(f'Post-cutoff events : {canonical_df["is_post_cutoff"].sum():,}  (excluded from model inputs)')

Learner × task cutoffs computed: 81
           academy_member_id  task_code                 cutoff_timestamp
0  MOCK_VALID3_20260715_S002  LQT000001 2026-07-14 20:21:28.538000+00:00
1  MOCK_VALID3_20260715_S002  LQT000002 2026-07-14 20:20:13.435000+00:00
2  MOCK_VALID3_20260715_S002  LQT000004 2026-07-14 20:20:45.443000+00:00
3  MOCK_VALID3_20260715_S002  LQT000005 2026-07-14 20:21:17.546000+00:00
4  MOCK_VALID3_20260715_S002  LQT000006 2026-07-14 20:20:24.373000+00:00

Pre-cutoff events  : 540
Post-cutoff events : 162  (excluded from model inputs)


## 4. Anti-leakage check

In [6]:
feature_cols = set(pre_cutoff_df.columns)
leaked = sorted(feature_cols & LEAKAGE_COLS)
if leaked:
    raise ValueError(f'LEAKAGE DETECTED — blacklisted columns in pre-cutoff data: {leaked}')
print('Anti-leakage check: PASS — no blacklisted columns in pre-cutoff event data')

Anti-leakage check: PASS — no blacklisted columns in pre-cutoff event data


## 5. Build per-step feature vectors

10-dimensional per-event feature vector. All features derived from pre-cutoff events only.

In [ ]:
# ── Phase 5 M5.2: Vocabulary — load from frozen artifact ─────────────────────
# PROBLEM: the previous dynamic build sorted observed event_type values and
# assigned tokens 1-N in alphabetical order. If Phase 5 block events appear,
# 'block_add' sorts before 'session_end' and receives token 1 instead of 6.
# This silently corrupts the sequence encoding for any mixed Phase 4+5 dataset.
#
# FIX: load the frozen vocabulary from lib/research-artifacts/phase4/vocabulary_v1.json
# (canonical token assignments for all 9 event types, including the reserved block
# slots that are now active in Phase 5). If the artifact is unreachable, fall back
# to dynamic building with a warning.

_frozen_paths = [
    Path('../lib/research-artifacts/phase4/vocabulary_v1.json'),
    Path('../../lib/research-artifacts/phase4/vocabulary_v1.json'),
]
_frozen_vocab_file = next((p for p in _frozen_paths if p.exists()), None)

if _frozen_vocab_file:
    with open(_frozen_vocab_file) as _fv:
        _frozen = json.load(_fv)
    vocab = _frozen['event_type_vocab']
    print(f'✅ Vocabulary loaded from frozen artifact: {_frozen_vocab_file}')
    print(f'   schema_version: {_frozen["schema_version"]}')
else:
    # Fallback: dynamic build — block events may get wrong token codes
    all_types  = sorted(canonical_df['event_type'].dropna().unique().tolist())
    reserved   = sorted(BLOCK_EVENT_TYPES - set(all_types))
    vocab_list = all_types + reserved
    vocab      = {et: i + 1 for i, et in enumerate(vocab_list)}
    print(f'⚠️  Frozen vocabulary artifact not found — built dynamically.')
    print(f'   Block event tokens may be incorrect for Phase 5 data.')

# Count active block events in this dataset (block_submit is not emitted)
block_found = int(canonical_df['event_type'].isin(BLOCK_EVENT_ACTIVE_TYPES).sum())

print(f'\nVocabulary size : {len(vocab)} event types  (0 = padding)')
for et, code in sorted(vocab.items(), key=lambda x: x[1]):
    if et in BLOCK_EVENT_ACTIVE_TYPES:
        phase_note = '  ← ACTIVE Phase 5'
    elif et in BLOCK_EVENT_RESERVED_TYPES:
        phase_note = '  ← RESERVED (not activated)'
    else:
        phase_note = ''
    in_data = ' ✓' if et in canonical_df['event_type'].values else ''
    print(f'  {code:3d}  {et}{phase_note}{in_data}')

print(f'\nBlock events in this dataset : {block_found}')
if block_found > 0:
    print(f'  ✅ Phase 5 block events present — tokens 6/7/8 applied from frozen vocab')
else:
    print(f'  ℹ️  No block events (Phase 4 data) — tokens 6/7/8 present in vocab, unused')

In [8]:
attempt_df['attempt_no'] = pd.to_numeric(attempt_df['attempt_no'], errors='coerce')
attempt_correct = attempt_df[['academy_member_id','task_code','attempt_no','is_correct']].copy()
attempt_correct['is_correct'] = pd.to_numeric(attempt_correct['is_correct'], errors='coerce')

records = []
for (learner_id, task_code), grp in pre_cutoff_df.groupby(['academy_member_id','task_code'], sort=False):
    grp = grp.sort_values('event_order').reset_index(drop=True)
    dur_max = pd.to_numeric(grp['duration_from_start'], errors='coerce').max()
    dur_max = float(dur_max) if (dur_max and dur_max > 0) else 1.0
    cum_run = cum_submit = cum_error = 0
    for _, row in grp.iterrows():
        et  = row['event_type']
        dur = float(pd.to_numeric(row['duration_from_start'], errors='coerce') or 0.0)
        is_run = int(et == 'sql_run');  is_sub = int(et == 'submit_answer');  is_err = int(et == 'sql_error')
        cum_run += is_run;  cum_submit += is_sub;  cum_error += is_err
        records.append({
            'academy_member_id':        learner_id,
            'task_code':                task_code,
            'event_id':                 row.get('event_id', ''),
            'event_order':              int(row['event_order']),
            'event_type':               et,
            'event_time':               row['event_time'],
            'event_type_code':          vocab.get(et, 0),
            'duration_from_start_norm': round(dur / dur_max, 6),
            'is_sql_run':               is_run,
            'is_submit':                is_sub,
            'is_error_event':           is_err,
            'cumulative_run_count':     cum_run,
            'cumulative_submit_count':  cum_submit,
            'cumulative_error_count':   cum_error,
            'attempt_is_correct':       np.nan,
            'step_position_norm':       0.0,
        })

feature_df = pd.DataFrame(records)
print(f'Feature rows (pre-cutoff steps): {len(feature_df):,}')

Feature rows (pre-cutoff steps): 540

In [9]:
for (lid, tc), grp in feature_df.groupby(['academy_member_id','task_code']):
    sub_idx = grp.index[grp['event_type'] == 'submit_answer']
    for rank, idx in enumerate(sub_idx, start=1):
        m = attempt_correct[
            (attempt_correct['academy_member_id'] == lid) &
            (attempt_correct['task_code'] == tc) &
            (attempt_correct['attempt_no'] == rank)
        ]
        if not m.empty:
            feature_df.at[idx, 'attempt_is_correct'] = float(m.iloc[0]['is_correct'])

seq_lengths = feature_df.groupby(['academy_member_id','task_code']).size()
max_len_raw = max(int(np.percentile(seq_lengths.values, MAX_SEQ_LEN_PERCENTILE)), 1)

for (lid, tc), grp in feature_df.groupby(['academy_member_id','task_code']):
    n = len(grp)
    feature_df.loc[grp.index, 'step_position_norm'] = [round(i/max(n-1,1), 6) for i in range(n)]

import matplotlib
matplotlib.use('Agg')  # non-interactive backend for nbconvert
import matplotlib.pyplot as plt
Path('../reports/phase4').mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(seq_lengths.values, bins=20, edgecolor='black')
ax.axvline(max_len_raw, color='red', linestyle='--', label=f'P{MAX_SEQ_LEN_PERCENTILE}={max_len_raw}')
ax.set_xlabel('Sequence length (steps)');  ax.set_ylabel('Count')
ax.set_title('Sequence length distribution (learner × task)');  ax.legend()
plt.tight_layout()
plt.savefig('../reports/phase4/seq_length_dist.png', dpi=150);  plt.close()
print(f'Seq lengths — min:{seq_lengths.min()}  median:{seq_lengths.median():.0f}  '
      f'P{MAX_SEQ_LEN_PERCENTILE}:{max_len_raw}  max:{seq_lengths.max()}')

Seq lengths — min:6  median:6  P95:6  max:6


## 6. Validate labels — priority rules + behavioral proxy fallback

**Label priority (research contract §5):**
1. `teacher_reviewed` — thesis valid
2. `expert_validated` — thesis valid
3. `auto_scored_validated` — thesis valid with caveat
4. `auto_generated` / `proxy_behavioral` — pipeline validation only
5. `no_rubric` / `unlabeled` — excluded from training

When all outcomes are `no_rubric`, a behavioral proxy is derived from the
attempt stream: `at_risk=1` if the learner never submitted a correct answer,
`at_risk=0` otherwise. This proxy is labeled `proxy_behavioral` /
`pilot_only` and is **not valid for thesis conclusions**.

In [10]:
outcome_df['total_2c3l_score'] = pd.to_numeric(outcome_df['total_2c3l_score'], errors='coerce')
outcome_df['at_risk']          = pd.to_numeric(outcome_df['at_risk'],          errors='coerce')

print('Label source distribution:')
print(outcome_df['label_source'].value_counts().to_string())

# Learner-level label: worst outcome across all tasks per learner
learner_labels = (
    outcome_df.sort_values('total_2c3l_score', ascending=True)
    .groupby('academy_member_id', as_index=False).first()
    [['academy_member_id','at_risk','label_source','label_validity','total_2c3l_score','is_teacher_reviewed']]
)
learner_labels['thesis_eligible'] = learner_labels['label_source'].isin(VALID_LABEL_SOURCES)
learner_labels['pilot_only']      = learner_labels['label_source'].isin(PILOT_LABEL_SOURCES)

# ── Behavioral proxy fallback ─────────────────────────────────────────────────
# When all outcomes are no_rubric, at_risk is NaN everywhere → derive from attempt stream.
all_no_rubric = learner_labels['at_risk'].isna().all()

if all_no_rubric:
    print('\n⚠️  All outcomes are no_rubric — no rubric scores in this batch.')
    print('   Deriving behavioral proxy: at_risk=1 if learner never submitted a correct answer.')
    print('   label_source=proxy_behavioral / label_validity=pilot_only')
    print('   This proxy is NOT valid for thesis conclusions.\n')

    attempt_correct_flag = attempt_df.copy()
    attempt_correct_flag['is_correct_bool'] = pd.to_numeric(
        attempt_correct_flag['is_correct'], errors='coerce'
    ).fillna(0).astype(bool)

    learner_any_correct = (
        attempt_correct_flag
        .groupby('academy_member_id')['is_correct_bool']
        .any()
        .reset_index()
        .rename(columns={'is_correct_bool': 'any_correct'})
    )

    # Use all learners present in the sequence data as the population
    all_seq_learners = pd.DataFrame(
        {'academy_member_id': canonical_df['academy_member_id'].unique()}
    )
    all_seq_learners = all_seq_learners.merge(learner_any_correct, on='academy_member_id', how='left')
    all_seq_learners['any_correct'] = all_seq_learners['any_correct'].fillna(False)
    all_seq_learners['at_risk']     = (~all_seq_learners['any_correct']).astype(int)

    learner_labels = all_seq_learners[['academy_member_id','at_risk']].copy()
    learner_labels['label_source']       = 'proxy_behavioral'
    learner_labels['label_validity']     = 'pilot_only'
    learner_labels['total_2c3l_score']   = np.nan
    learner_labels['is_teacher_reviewed']= False
    learner_labels['thesis_eligible']    = False
    learner_labels['pilot_only']         = True

print(f'\nLearner labels:')
print(f'  Total                : {len(learner_labels)}')
print(f'  thesis_eligible      : {learner_labels["thesis_eligible"].sum()}')
print(f'  pilot_only           : {learner_labels["pilot_only"].sum()}')
print(f'  at_risk=1            : {(learner_labels["at_risk"] == 1).sum()}')
print(f'  at_risk=0            : {(learner_labels["at_risk"] == 0).sum()}')
print(f'  no label (NaN)       : {learner_labels["at_risk"].isna().sum()}')
if learner_labels['thesis_eligible'].sum() == 0:
    print('\n⚠️  WARNING: No thesis-eligible labels — pipeline validation only.')

Label source distribution:
label_source
no_rubric    81

⚠️  All outcomes are no_rubric — no rubric scores in this batch.
   Deriving behavioral proxy: at_risk=1 if learner never submitted a correct answer.
   label_source=proxy_behavioral / label_validity=pilot_only
   This proxy is NOT valid for thesis conclusions.


Learner labels:
  Total                : 10
  thesis_eligible      : 0
  pilot_only           : 10
  at_risk=1            : 4
  at_risk=0            : 6
  no label (NaN)       : 0

⚠️  WARNING: No thesis-eligible labels — pipeline validation only.


## 7. Student-level split — GroupShuffleSplit, frozen ledger

In [11]:
eligible = learner_labels[learner_labels['at_risk'].notna()].copy().reset_index(drop=True)
print(f'Eligible learners for split: {len(eligible)}')

if len(eligible) < 3:
    raise ValueError(
        f'Only {len(eligible)} eligible learner(s). '
        'GroupShuffleSplit requires ≥2 groups per split. Collect more data.'
    )

gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(
    gss.split(np.zeros((len(eligible),1)), eligible['at_risk'].values, eligible['academy_member_id'].values)
)
eligible['split'] = 'train'
eligible.loc[test_idx, 'split'] = 'test'

train_learners = set(eligible.loc[train_idx, 'academy_member_id'])
test_learners  = set(eligible.loc[test_idx,  'academy_member_id'])
overlap        = train_learners & test_learners
assert not overlap, f'SPLIT LEAKAGE: {overlap}'

print(f'Train learners : {len(train_learners)}')
print(f'Test  learners : {len(test_learners)}')
print(f'Overlap        : {len(overlap)}  (must be 0)')
print(f'at_risk train  : {eligible[eligible["split"]=="train"]["at_risk"].value_counts().to_dict()}')
print(f'at_risk test   : {eligible[eligible["split"]=="test" ]["at_risk"].value_counts().to_dict()}')

Eligible learners for split: 10
Train learners : 8
Test  learners : 2
Overlap        : 0  (must be 0)
at_risk train  : {0: 5, 1: 3}
at_risk test   : {1: 1, 0: 1}


In [12]:
split_ledger = eligible[[
    'academy_member_id','split','at_risk',
    'label_source','label_validity','thesis_eligible',
    'total_2c3l_score','is_teacher_reviewed',
]].copy()

ledger_path = SEQ_DIR / 'split_assignments.parquet'
split_ledger.to_parquet(ledger_path, index=False)
print(f'Split ledger saved: {ledger_path}  ({len(split_ledger)} rows)')
display(split_ledger)

Split ledger saved: data\sequences\split_assignments.parquet  (10 rows)


,academy_member_id,split,at_risk,label_source,label_validity,thesis_eligible,total_2c3l_score,is_teacher_reviewed
0,MOCK_VALID3_20260715_S001,train,0,proxy_behavioral,pilot_only,False,NaN,False
1,MOCK_VALID3_20260715_S003,train,0,proxy_behavioral,pilot_only,False,NaN,False
2,MOCK_VALID3_20260715_S004,train,0,proxy_behavioral,pilot_only,False,NaN,False
3,MOCK_VALID3_20260715_S009,test,1,proxy_behavioral,pilot_only,False,NaN,False
4,MOCK_VALID3_20260715_S002,test,0,proxy_behavioral,pilot_only,False,NaN,False
5,MOCK_VALID3_20260715_S008,train,1,proxy_behavioral,pilot_only,False,NaN,False
6,MOCK_VALID3_20260715_S007,train,1,proxy_behavioral,pilot_only,False,NaN,False
7,MOCK_VALID3_20260715_S005,train,0,proxy_behavioral,pilot_only,False,NaN,False
8,MOCK_VALID3_20260715_S010,train,1,proxy_behavioral,pilot_only,False,NaN,False
9,MOCK_VALID3_20260715_S006,train,0,proxy_behavioral,pilot_only,False,NaN,False


## 8. Build sequence index and canonical events artifact

In [13]:
canonical_export = seq_df[[
    'academy_member_id','batch_code','task_code',
    'session_id','event_id','event_order',
    'event_type','event_value','duration_from_start',
    'event_time','dropped_as_duplicate',
]].copy()
canonical_export = canonical_export.merge(
    canonical_df[['event_id','cutoff_timestamp','is_post_cutoff']].drop_duplicates('event_id'),
    on='event_id', how='left',
)
canonical_export = canonical_export.merge(
    split_ledger[['academy_member_id','split']], on='academy_member_id', how='left',
)
canonical_path = SEQ_DIR / 'canonical_events.parquet'
canonical_export.to_parquet(canonical_path, index=False)
print(f'Canonical events saved: {canonical_path}  ({len(canonical_export)} rows)')

seq_index = (
    feature_df.groupby(['academy_member_id','task_code'], as_index=False)
    .agg(n_steps=('event_order','count'),
         first_event_time=('event_time','min'),
         last_event_time=('event_time','max'))
)
seq_index = seq_index.merge(cutoff_df, on=['academy_member_id','task_code'], how='left')
seq_index = seq_index.merge(
    split_ledger[['academy_member_id','split','at_risk','label_source','label_validity']],
    on='academy_member_id', how='left',
)
seq_index_path = SEQ_DIR / 'sequence_index.parquet'
seq_index.to_parquet(seq_index_path, index=False)
print(f'Sequence index saved: {seq_index_path}  ({len(seq_index)} rows)')
display(seq_index.head())

Canonical events saved: data\sequences\canonical_events.parquet  (702 rows)
Sequence index saved: data\sequences\sequence_index.parquet  (90 rows)


,academy_member_id,task_code,n_steps,first_event_time,last_event_time,cutoff_timestamp,split,at_risk,label_source,label_validity
0,MOCK_VALID3_20260715_S001,LQT000001,6,2026-07-14 20:20:54.969000+00:00,2026-07-14 20:21:00.922000+00:00,NaT,train,0,proxy_behavioral,pilot_only
1,MOCK_VALID3_20260715_S001,LQT000002,6,2026-07-14 20:20:02.775000+00:00,2026-07-14 20:20:08.647000+00:00,NaT,train,0,proxy_behavioral,pilot_only
2,MOCK_VALID3_20260715_S001,LQT000004,6,2026-07-14 20:20:25.846000+00:00,2026-07-14 20:20:31.641000+00:00,NaT,train,0,proxy_behavioral,pilot_only
3,MOCK_VALID3_20260715_S001,LQT000005,6,2026-07-14 20:20:47.837000+00:00,2026-07-14 20:20:53.387000+00:00,NaT,train,0,proxy_behavioral,pilot_only
4,MOCK_VALID3_20260715_S001,LQT000006,6,2026-07-14 20:20:10.284000+00:00,2026-07-14 20:20:16.183000+00:00,NaT,train,0,proxy_behavioral,pilot_only


## 9. Pad + mask sequences → tensors

Vocabulary and scaler are fit on **training data only**, then applied to test.

In [14]:
NUMERIC_FEATURES = [
    'event_type_code','duration_from_start_norm',
    'is_sql_run','is_submit','is_error_event',
    'cumulative_run_count','cumulative_submit_count','cumulative_error_count',
    'attempt_is_correct','step_position_norm',
]
N_FEATURES = len(NUMERIC_FEATURES)

split_map = dict(zip(split_ledger['academy_member_id'], split_ledger['split']))
label_map = dict(zip(split_ledger['academy_member_id'], split_ledger['at_risk']))
feature_df['split'] = feature_df['academy_member_id'].map(split_map)

learner_task_ids = (
    feature_df[feature_df['split'].notna()]
    .groupby(['academy_member_id','task_code']).size().reset_index()
    [['academy_member_id','task_code']].values.tolist()
)

train_steps = feature_df[feature_df['split'] == 'train'][NUMERIC_FEATURES].copy()
train_steps['attempt_is_correct'] = train_steps['attempt_is_correct'].fillna(0.0)
scaler = StandardScaler()
scaler.fit(train_steps)

print(f'Scaler fit on {len(train_steps)} training steps')
print(f'Max sequence length (P{MAX_SEQ_LEN_PERCENTILE}): {max_len_raw} steps')
print(f'Feature dimensions: {N_FEATURES}')

Scaler fit on 432 training steps
Max sequence length (P95): 6 steps
Feature dimensions: 10


In [15]:
def build_tensors(split_name: str) -> tuple:
    learners = {lid for lid, sp in split_map.items() if sp == split_name}
    pairs    = [(lid, tc) for lid, tc in learner_task_ids if lid in learners]
    X    = np.zeros((len(pairs), max_len_raw, N_FEATURES), dtype=np.float32)
    mask = np.zeros((len(pairs), max_len_raw),             dtype=bool)
    y    = np.zeros(len(pairs),                            dtype=np.int8)
    ids  = []
    for i, (lid, tc) in enumerate(pairs):
        grp = feature_df[
            (feature_df['academy_member_id'] == lid) & (feature_df['task_code'] == tc)
        ].sort_values('event_order')[NUMERIC_FEATURES].copy()
        grp['attempt_is_correct'] = grp['attempt_is_correct'].fillna(0.0)
        steps = scaler.transform(grp.values)
        n     = min(len(steps), max_len_raw)
        X[i, :n, :] = steps[:n]
        mask[i, :n]  = True
        y[i]         = int(label_map.get(lid, 0))
        ids.append(f'{lid}::{tc}')
    return X, y, mask, ids

X_train, y_train, mask_train, ids_train = build_tensors('train')
X_test,  y_test,  mask_test,  ids_test  = build_tensors('test')

print(f'Train : X={X_train.shape}  y={y_train.shape}  mask={mask_train.shape}')
print(f'Test  : X={X_test.shape}   y={y_test.shape}   mask={mask_test.shape}')
print(f'Train at_risk : {dict(zip(*np.unique(y_train, return_counts=True)))}')
print(f'Test  at_risk : {dict(zip(*np.unique(y_test,  return_counts=True)))}')

C:\Users\n.sukkhadamrongrak\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\n.sukkhadamrongrak\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\n.sukkhadamrongrak\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\n.sukkhadamrongrak\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\n.sukkhadamrongrak\AppData\Local\Python\pythonc

Train : X=(72, 6, 10)  y=(72,)  mask=(72, 6)
Test  : X=(18, 6, 10)   y=(18,)   mask=(18, 6)
Train at_risk : {np.int8(0): np.int64(45), np.int8(1): np.int64(27)}
Test  at_risk : {np.int8(0): np.int64(9), np.int8(1): np.int64(9)}


C:\Users\n.sukkhadamrongrak\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [16]:
assert X_train.ndim == 3
assert X_train.shape[1] == max_len_raw
assert X_train.shape[2] == N_FEATURES
assert X_train.shape[0] == mask_train.shape[0]
assert mask_train.shape[1] == max_len_raw
assert not np.isnan(X_train).any(), 'X_train contains NaN'
assert not np.isnan(X_test).any(),  'X_test contains NaN'
print('Tensor shape + NaN validation: PASS')

Tensor shape + NaN validation: PASS


## 10. Write artifacts and sequence manifest

In [ ]:
tensors_path = SEQ_DIR / 'sequence_tensors_v1.npz'
np.savez_compressed(tensors_path,
    X_train=X_train, y_train=y_train, mask_train=mask_train,
    X_test=X_test,   y_test=y_test,   mask_test=mask_test)
print(f'Tensors saved: {tensors_path}')

# Phase 5 M5.2: vocabulary note updated — block events active, not reserved
vocab_note = (
    'Phase 5 M5.2: block_add=6, block_delete=7, block_move=8 are ACTIVE '
    '(emitted via POST /api/student/block-event). '
    'block_submit=9 is RESERVED and not activated '
    '(see PHASE5_BLOCK_EVENT_CONTRACT_v1.md §3). '
    f'Block events in this run: {block_found}.'
)
vocab_path = SEQ_DIR / 'vocabulary_v1.json'
vocab_path.write_text(json.dumps({
    'schema_version':          SCHEMA_VERSION,
    'padding_token':           0,
    'event_type_vocab':        vocab,
    'block_events_active':     sorted(BLOCK_EVENT_ACTIVE_TYPES),
    'block_events_reserved':   sorted(BLOCK_EVENT_RESERVED_TYPES),
    'block_found_in_run':      block_found,
    'note':                    vocab_note,
}, indent=2))
print(f'Vocabulary saved: {vocab_path}')

scaler_path = SEQ_DIR / 'scaler_v1.json'
scaler_path.write_text(json.dumps({
    'schema_version': SCHEMA_VERSION, 'feature_names': NUMERIC_FEATURES,
    'mean_': scaler.mean_.tolist(), 'scale_': scaler.scale_.tolist(),
    'n_samples_seen': int(scaler.n_samples_seen_), 'fit_split': 'train',
}, indent=2))
print(f'Scaler saved: {scaler_path}')

In [18]:
manifest = {
    'schema_version':    SCHEMA_VERSION,
    'created_at_utc':    datetime.now(timezone.utc).isoformat(),
    'phase3_source_sha': PHASE3_SOURCE_SHA,
    'input_files': {
        'sequence_csv': str(seq_path),  'sequence_sha': sha256_file(seq_path),
        'attempt_csv':  str(attempt_path), 'attempt_sha':  sha256_file(attempt_path),
        'outcome_csv':  str(outcome_path), 'outcome_sha':  sha256_file(outcome_path),
    },
    'parameters': {
        'at_risk_threshold': AT_RISK_THRESHOLD, 'dedup_window_sec': DEDUP_WINDOW_SEC,
        'max_seq_len_percentile': MAX_SEQ_LEN_PERCENTILE, 'max_seq_len': max_len_raw,
        'n_features': N_FEATURES, 'feature_names': NUMERIC_FEATURES,
        'random_state': RANDOM_STATE, 'test_size': TEST_SIZE,
    },
    'dataset_stats': {
        'raw_events':              int(len(seq_df)),
        'dropped_as_duplicate':    n_dropped,
        'canonical_events':        int(len(canonical_df)),
        'pre_cutoff_events':       int(len(pre_cutoff_df)),
        'total_learners':          int(len(learner_labels)),
        'eligible_learners':       int(len(eligible)),
        'train_learners':          int(len(train_learners)),
        'test_learners':           int(len(test_learners)),
        'thesis_eligible_labels':  int(learner_labels['thesis_eligible'].sum()),
        'proxy_behavioral_labels': int((learner_labels['label_source'] == 'proxy_behavioral').sum()),
        'train_shape':             list(X_train.shape),
        'test_shape':              list(X_test.shape),
    },
    'data_warning': (
        'TECHNICAL VALIDATION ONLY — mock dataset, no valid 2C3L labels. '
        'labels=proxy_behavioral/pilot_only. Final thesis requires >=60 participants.'
    ),
    'artifacts': {
        'canonical_events':  str(canonical_path),
        'sequence_index':    str(seq_index_path),
        'split_assignments': str(ledger_path),
        'sequence_tensors':  str(tensors_path),
        'vocabulary':        str(vocab_path),
        'scaler':            str(scaler_path),
    },
}
manifest_path = SEQ_DIR / 'sequence_manifest_v1.json'
manifest_path.write_text(json.dumps(manifest, indent=2, default=str))
print(f'Manifest saved: {manifest_path}')
print(json.dumps(manifest['dataset_stats'], indent=2))

Manifest saved: data\sequences\sequence_manifest_v1.json
{
  "raw_events": 702,
  "dropped_as_duplicate": 0,
  "canonical_events": 702,
  "pre_cutoff_events": 540,
  "total_learners": 10,
  "eligible_learners": 10,
  "train_learners": 8,
  "test_learners": 2,
  "thesis_eligible_labels": 0,
  "proxy_behavioral_labels": 10,
  "train_shape": [
    72,
    6,
    10
  ],
  "test_shape": [
    18,
    6,
    10
  ]
}


## 11. Validation summary — 13 checks

In [ ]:
checks = []
def chk(name, passed, detail=''):
    checks.append({'check': name, 'result': 'PASS' if passed else 'FAIL', 'detail': detail})

# Phase 5 M5.2: validate block event tokens instead of asserting absence.
# For each active block event type that appears in the dataset, verify it
# maps to the canonical token code (block_add=6, delete=7, move=8).
_expected_block_tokens = {'block_add': 6, 'block_delete': 7, 'block_move': 8}
_block_token_errors    = [
    f'{et}: expected {exp}, got {vocab.get(et, "missing")}'
    for et, exp in _expected_block_tokens.items()
    if vocab.get(et) != exp
]
block_token_ok = len(_block_token_errors) == 0

chk('Snapshot files loaded',           all(p is not None for p in [seq_path, attempt_path, outcome_path]))
chk('Required columns (all snapshots)', True, 'sequence / attempt / outcome')
chk('Event deduplication',             n_dropped >= 0, f'{n_dropped} client-side duplicates dropped')
chk('Block event tokens validated',    block_token_ok,
    (f'{block_found} block events; tokens correct' if block_token_ok
     else 'token mismatch: ' + '; '.join(_block_token_errors)))
chk('Cutoff timestamps computed',      len(cutoff_df) > 0, f'{len(cutoff_df)} learner×task cutoffs')
chk('Anti-leakage',                    len(leaked) == 0,
    'blacklisted: ' + (', '.join(leaked) if leaked else 'none'))
chk('Split: no learner overlap',       len(overlap) == 0, f'overlap={len(overlap)}')
chk('Tensor shape (3-D)',              X_train.ndim == 3 and X_test.ndim == 3,
    f'train={X_train.shape}  test={X_test.shape}')
chk('No NaN in tensors',               not np.isnan(X_train).any() and not np.isnan(X_test).any())
chk('Mask shape matches tensor',       mask_train.shape == X_train.shape[:2] and mask_test.shape == X_test.shape[:2])
chk('Split ledger saved',              ledger_path.exists())
chk('Manifest saved with checksums',   manifest_path.exists() and 'sequence_sha' in manifest['input_files'])
chk('Label validity documented',       'label_source' in split_ledger.columns,
    f'thesis_eligible={learner_labels["thesis_eligible"].sum()}  '
    f'pilot_only={learner_labels["pilot_only"].sum()}')

result_df = pd.DataFrame(checks)
n_fail    = (result_df['result'] == 'FAIL').sum()

print('\n── NB05 Validation Summary ───────────────────────────────────────────')
print(result_df.to_string(index=False))
print(f'\n{len(checks) - n_fail}/{len(checks)} checks passed')

if n_fail > 0:
    raise RuntimeError(f'NB05 validation FAILED — {n_fail} check(s) did not pass.')

print('\n✅ NB05 COMPLETE — artifacts ready for NB06 (TAG) and NB07 (LSTM).')
print(f'\n   Block events: {block_found} active events in this run '
      f'({"Phase 5 data" if block_found > 0 else "Phase 4 data — tokens reserved"})')
print('\n⚠️  TECHNICAL VALIDATION ONLY.')
print(f'   label_source={split_ledger["label_source"].iloc[0]} / '
      f'label_validity={split_ledger["label_validity"].iloc[0]}')
print(f'   Learners={len(eligible)}  (thesis requires ≥60)')